# U.S. State-Level Unemployment Rate Data Preparation and Panel Construction

This notebook prepares the monthly U.S. state unemployment rate dataset from raw BLS LAUS(Bureau of Labor Statistics – Local Area Unemployment Statistics) files into a clean State–Year–Month panel format. The output dataset is used as an explanatory variable input for structural break analysis in the Capstone Project.

In [1]:
import prettytable
import sqlite3
import requests
import pandas as pd
from pathlib import Path
prettytable.DEFAULT = 'DEFAULT'

## 1) Load raw LAUS file

In [2]:
# --- Resolve project root robustly ---
current = Path().resolve()

while current != current.parent:
    if (current / "Data").exists():
        PROJECT_ROOT = current
        break
    current = current.parent
else:
    raise FileNotFoundError("Project root with 'Data' folder not found")

DATA_RAW = PROJECT_ROOT / "Data" / "Raw"

# Load LAUS raw file
df = pd.read_csv(DATA_RAW / "la.data.2.AllStatesU.txt", sep="\t")

# Clean columns
df.columns = df.columns.str.strip()
df["series_id"] = df["series_id"].str.strip()


## 2) Filter unemployment rate series and valid months

In [3]:
#Keep unemployment rate only (003)
df = df[df["series_id"].str[-3:] == "003"].copy()

#Remove annual averages (M13)
df = df[df["period"] != "M13"]

## 3) Extract state identifiers and time variables

In [4]:
# Extract FIPS
df["state_fips"] = df["series_id"].str[5:7]

# Convert numeric
df["year"] = df["year"].astype(int)
df["value"] = pd.to_numeric(df["value"], errors="coerce")

# Extract month
df["month"] = df["period"].str[1:].astype(int)

## 4) Construct monthly date index

In [5]:
# Create date
df["date"] = pd.to_datetime(dict(year=df["year"], month=df["month"], day=1))

## 5) Map FIPS codes to state names

In [6]:
# FIPS → State (50 states only)
fips_to_state = {
    "01":"Alabama","02":"Alaska","04":"Arizona","05":"Arkansas",
    "06":"California","08":"Colorado","09":"Connecticut",
    "10":"Delaware","12":"Florida","13":"Georgia","15":"Hawaii",
    "16":"Idaho","17":"Illinois","18":"Indiana","19":"Iowa",
    "20":"Kansas","21":"Kentucky","22":"Louisiana","23":"Maine",
    "24":"Maryland","25":"Massachusetts","26":"Michigan",
    "27":"Minnesota","28":"Mississippi","29":"Missouri",
    "30":"Montana","31":"Nebraska","32":"Nevada",
    "33":"New Hampshire","34":"New Jersey","35":"New Mexico",
    "36":"New York","37":"North Carolina","38":"North Dakota",
    "39":"Ohio","40":"Oklahoma","41":"Oregon","42":"Pennsylvania",
    "44":"Rhode Island","45":"South Carolina","46":"South Dakota",
    "47":"Tennessee","48":"Texas","49":"Utah","50":"Vermont",
    "51":"Virginia","53":"Washington","54":"West Virginia",
    "55":"Wisconsin","56":"Wyoming"
}

df["state"] = df["state_fips"].map(fips_to_state)

df = df[df["state"].notna()].copy()

## 6) Restrict analysis period (2016–2025)

In [7]:
# Filter years
df = df[(df["year"] >= 2016) & (df["year"] <= 2025)]

## 7) Build final unemployment panel

In [8]:
df_final = df[["state","year","month","value"]].copy()

df_final.rename(columns={"value":"unemployment_rate"}, inplace=True)

df_final = df_final.sort_values(["state","year"]).reset_index(drop=True)

# REMOVE IDAHO and Wyoming
df_final = df_final[df_final["state"] != "Idaho"].copy()
df_final = df_final[df_final["state"] != "Wyoming"].copy()

## 8) Inspect panel structure

In [9]:
print("States:", df_final["state"].nunique())
print("Rows:", len(df_final))
print(df_final.head())

States: 48
Rows: 5760
     state  year  month  unemployment_rate
0  Alabama  2016      1                6.3
1  Alabama  2016      2                6.2
2  Alabama  2016      3                5.9
3  Alabama  2016      4                5.4
4  Alabama  2016      5                5.3


## 9) Handle missing values (state-wise interpolation)

In [10]:
df_final.isna().sum()

state                 0
year                  0
month                 0
unemployment_rate    48
dtype: int64

In [11]:
df_final = df_final.sort_values(["state", "year"])

df_final["unemployment_rate"] = (
    df_final.groupby("state")["unemployment_rate"]
    .transform(lambda x: x.interpolate(method="linear"))
)

# Fill remaining edge missing values
df_final["unemployment_rate"] = (
    df_final.groupby("state")["unemployment_rate"]
    .transform(lambda x: x.ffill().bfill())
)


## 10) Verify missingness

In [12]:
df_final.isna().sum()

state                0
year                 0
month                0
unemployment_rate    0
dtype: int64

## 11) Export clean unemployment panel 

Merge keys you will use later:

- `State`, `Year`, `Month`

This will allow clean merges with:

- Motor bus ridership (state–month)
- policy stringency/mobility (state–month)
- fuel prices (region/state)
- red/blue classification (state-level)
- workforce composition (state-year)
- GDP (State/Year)

In [13]:
# Resolve project root
current = Path().resolve()
while current != current.parent:
    if (current / "Data").exists():
        PROJECT_ROOT = current
        break
    current = current.parent

DATA_CLEAN = PROJECT_ROOT / "Data" / "Cleaned"

# Save cleaned dataset

EXCLUDED_STATES = ["Idaho", "Wyoming"]
df_final = df_final[~df_final["state"].isin(EXCLUDED_STATES)]


output_path = DATA_CLEAN / "state_unemployment_rate_monthly_2016_2025.csv"
df_final.to_csv(output_path, index=False)

print(f"Saved: {output_path}")

Saved: /Users/emerino/Documents/Capstone_Project_Group_4-/Data/Cleaned/state_unemployment_rate_monthly_2016_2025.csv


## 12) Load panel into SQLite database

In [14]:
# --- Connect to SAME SQLite file ---
db_path = PROJECT_ROOT / "Capstone.db"

con = sqlite3.connect(db_path)

%load_ext sql
%sql sqlite:///{db_path}

# --- Read cleaned CSV ---
df = pd.read_csv(DATA_CLEAN / "state_unemployment_rate_monthly_2016_2025.csv")

# --- Remove excluded states (safety) ---
EXCLUDED_STATES = ["Idaho", "Wyoming"]
df = df[~df["state"].isin(EXCLUDED_STATES)]

print("States after filter:", df["state"].nunique())  # should be 48

# --- Write to SQLite ---
df.to_sql(
    "Unemployment",
    con,
    if_exists="replace",
    index=False,
    method="multi"
)

print("Unemployment table overwritten")


States after filter: 48
Unemployment table overwritten


## 13) Validate database ingestion

In [15]:
%sql SELECT * FROM Unemployment LIMIT 5;

 * sqlite:////Users/emerino/Documents/Capstone_Project_Group_4-/Capstone.db
Done.


state,year,month,unemployment_rate
Alabama,2016,1,6.3
Alabama,2016,2,6.2
Alabama,2016,3,5.9
Alabama,2016,4,5.4
Alabama,2016,5,5.3


In [16]:
%%sql SELECT COUNT(DISTINCT state) AS num_states
FROM Unemployment;

 * sqlite:////Users/emerino/Documents/Capstone_Project_Group_4-/Capstone.db
Done.


num_states
48


In [17]:
%%sql SELECT DISTINCT state
FROM Unemployment
ORDER BY state;

 * sqlite:////Users/emerino/Documents/Capstone_Project_Group_4-/Capstone.db
Done.


state
Alabama
Alaska
Arizona
Arkansas
California
Colorado
Connecticut
Delaware
Florida
Georgia
